# Caching & Performance

DataCore ships with a built-in `ParquetCache` that persists pulled DataFrames to disk. For research workflows you re-run dozens of times a day, this turns a 15-second network round-trip into a 50ms parquet read.

**Why this matters**: research notebooks are re-executed constantly. Without caching, you waste quota and time on data that hasn't changed. With caching, the second run of any backtest is effectively instant.

## Enabling the cache

The cache is opt-in. Pass a `cache=` argument when constructing the client, or wire one up explicitly with `ParquetCache`.

In [ ]:
from pathlib import Path
from datacore import Client
from datacore.cache import ParquetCache

cache_dir = Path.home() / '.datacore' / 'cache'
cache = ParquetCache(root=cache_dir, max_size_gb=5)

dc = Client(cache=cache)
print('Cache root:', cache.root)
print('Current size:', f'{cache.size_bytes / 1e6:.1f} MB')

## Comparing pull time with vs without cache

Let's time the same query twice. The first call goes over the wire; the second hits the parquet on disk.

In [ ]:
import time

def timed_pull(label):
    t0 = time.perf_counter()
    df = dc.dataset('equity.vn30.daily').to_pandas(
        start='2020-01-01', end='2024-12-31'
    )
    dt = time.perf_counter() - t0
    print(f'{label}: {len(df):,} rows in {dt*1000:.0f} ms')
    return df

_ = timed_pull('Cold (network)')
_ = timed_pull('Warm (cache hit)')

On a typical run you'll see something like:

```
Cold (network): 8,210 rows in 4,830 ms
Warm (cache hit): 8,210 rows in 41 ms
```

A ~100x speedup, and zero quota consumed on the warm call.

## Manual invalidation

Sometimes you need to bust the cache — a corporate action restated history, you suspect stale data, or you want fresh EOD prices after the close. Three options:

In [ ]:
# Option 1: bypass cache for a single call
df = dc.dataset('equity.vn30.daily').to_pandas(
    start='2024-12-01', end='2024-12-31',
    use_cache=False,
)

# Option 2: invalidate one dataset
cache.invalidate('equity.vn30.daily')

# Option 3: nuke everything
# cache.clear()

print('Cache size after invalidate:', f'{cache.size_bytes / 1e6:.1f} MB')

## TTL-based freshness

For datasets that update daily (EOD prices, fundamentals), a TTL avoids stale reads without forcing manual invalidation. Set it per-dataset:

In [ ]:
from datetime import timedelta

# Re-pull anything older than 12 hours
df = dc.dataset('equity.vn30.daily').to_pandas(
    start='2024-01-01',
    cache_ttl=timedelta(hours=12),
)
df.tail(3)

## When to cache — and when not to

**Cache aggressively when**:
- Running backtests (same history, dozens of parameter sweeps)
- Notebook research (re-execute cell 50 times while iterating)
- Cross-sectional studies that touch hundreds of tickers
- Anything with a fixed historical window

**Don't cache when**:
- Intraday strategies that need fresh ticks
- Live signal generation in production
- Pre-market or post-close runs where the dataset is being updated
- Real-time order book or trade tape

In [ ]:
# Intraday workflows: explicitly opt out
live_dc = Client(cache=None)

tick = live_dc.dataset('equity.intraday.1min').to_pandas(
    ticker='VNM',
    start='2025-05-29 09:00',
    end='2025-05-29 11:30',
)
tick.tail()

## Inspecting what's cached

Good housekeeping: list cached entries and their ages so you know what's eating your disk.

In [ ]:
for entry in cache.list():
    print(f"{entry['dataset_id']:30s}  {entry['size_mb']:6.1f} MB  age={entry['age_hours']:.1f}h")

## Next steps

- `02-equity-research/01-vn30-fundamentals.ipynb` — first real screener (uses cache)
- `03-factor-investing/01-fama-french-vn.ipynb` — heavy backtest, cache pays for itself in minutes
- `04-alternative-data/01-ecommerce-signals.ipynb` — cache big alt-data pulls